<a href="https://colab.research.google.com/github/rymarinelli/Bayes-Benchmarking/blob/main/Bayesian_Benchmarking_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Benchmarking
Task-level sequential comparison of NorMistral-11B vs NorMistral-7B on NorEval (nob_world).

**Upload your zip files first** using the cell below, then run all cells in order.

In [1]:
# Upload Model Results
from google.colab import files

print("Upload your two zip files (7b_results.zip and noreval_results_11B.zip)")
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Upload your two zip files (7b_results.zip and noreval_results_11B.zip)


Saving noreval_results_11B.zip to noreval_results_11B.zip
Saving 7b_results.zip to 7b_results.zip
Uploaded: ['noreval_results_11B.zip', '7b_results.zip']


In [2]:
# Extract zips
import zipfile, os

os.makedirs("noreval_results/7b",  exist_ok=True)
os.makedirs("noreval_results/11b", exist_ok=True)

for fname in uploaded.keys():
    label = "7b" if "7b" in fname.lower() else "11b"
    with zipfile.ZipFile(fname) as z:
        z.extractall(f"noreval_results/{label}")
    print(f"Extracted {fname} → noreval_results/{label}/")

Extracted noreval_results_11B.zip → noreval_results/11b/
Extracted 7b_results.zip → noreval_results/7b/


In [3]:
# Install dependencies
!pip install scipy numpy --quiet
print("Ready.")

Ready.


In [4]:
#Load samples from jsonl files
import json, glob
from pathlib import Path
from collections import defaultdict

def load_samples(output_dir: str) -> dict:
    results = defaultdict(list)
    pattern = str(Path(output_dir) / "**" / "samples_*.jsonl")
    files_found = sorted(glob.glob(pattern, recursive=True))

    if not files_found:
        raise FileNotFoundError(
            f"No samples_*.jsonl files found under {output_dir}.\n"
            "Make sure lm_eval ran with --log_samples and --write_out."
        )

    for fpath in files_found:
        task = Path(fpath).stem.split("_p")[0].replace("samples_", "")
        with open(fpath) as f:
            for line in f:
                row = json.loads(line)
                acc = row.get("acc", row.get("exact_match", 0))
                results[task].append(float(acc) > 0.5)

    return dict(results)


samples_7b  = load_samples("noreval_results/7b")
samples_11b = load_samples("noreval_results/11b")

print("=== 7B ===")
for task, outcomes in samples_7b.items():
    print(f"  {task}: {len(outcomes)} samples, {sum(outcomes)} correct ({sum(outcomes)/len(outcomes):.1%})")

print("\n=== 11B ===")
for task, outcomes in samples_11b.items():
    print(f"  {task}: {len(outcomes)} samples, {sum(outcomes)} correct ({sum(outcomes)/len(outcomes):.1%})")

=== 7B ===
  norbelebele: 4500 samples, 1231 correct (27.4%)
  norcommonsenseqa_nob: 4990 samples, 1513 correct (30.3%)
  noropenbookqa_nob: 1870 samples, 705 correct (37.7%)
  nortruthfulqa_mc_nob: 2440 samples, 990 correct (40.6%)
  nrk_quiz_qa_nob: 18000 samples, 7407 correct (41.1%)

=== 11B ===
  norbelebele: 4500 samples, 2188 correct (48.6%)
  norcommonsenseqa_nob: 4990 samples, 3306 correct (66.3%)
  noropenbookqa_nob: 1870 samples, 1321 correct (70.6%)
  nortruthfulqa_mc_nob: 2440 samples, 1353 correct (55.5%)
  nrk_quiz_qa_nob: 18000 samples, 10531 correct (58.5%)


In [5]:
# Bayesian core
import numpy as np
from scipy import integrate, stats
from dataclasses import dataclass


@dataclass
class BetaPosterior:
    alpha_prior: float = 1.0
    beta_prior:  float = 1.0
    successes: int = 0
    trials:    int = 0

    def observe_one(self, outcome: bool):
        self.trials += 1
        if outcome:
            self.successes += 1

    @property
    def alpha(self): return self.alpha_prior + self.successes

    @property
    def beta(self): return self.beta_prior + (self.trials - self.successes)

    @property
    def mean(self): return self.alpha / (self.alpha + self.beta)

    def credible_interval(self, ci=0.95):
        lo = (1 - ci) / 2
        return stats.beta.ppf(lo, self.alpha, self.beta), stats.beta.ppf(1 - lo, self.alpha, self.beta)


def prob_a_beats_b(post_a: BetaPosterior, post_b: BetaPosterior) -> float:
    integrand = lambda x: (
        stats.beta.pdf(x, post_a.alpha, post_a.beta) *
        stats.beta.cdf(x, post_b.alpha, post_b.beta)
    )
    result, _ = integrate.quad(integrand, 0, 1, limit=100)
    return float(np.clip(result, 0, 1))


def is_non_discriminating(post_a: BetaPosterior, post_b: BetaPosterior,
                           threshold=0.85) -> bool:
    p_a_easy = 1 - stats.beta.cdf(threshold, post_a.alpha, post_a.beta)
    p_b_easy = 1 - stats.beta.cdf(threshold, post_b.alpha, post_b.beta)
    p_a_hard = stats.beta.cdf(1 - threshold, post_a.alpha, post_a.beta)
    p_b_hard = stats.beta.cdf(1 - threshold, post_b.alpha, post_b.beta)
    return (
        (p_a_easy > 0.9 and p_b_easy > 0.9) or
        (p_a_hard > 0.9 and p_b_hard > 0.9)
    )


print("Bayesian core loaded.")

Bayesian core loaded.


In [6]:
# Config
CONFIDENCE     = 0.95   # γ: stop when P(11B > 7B) crosses this threshold
SKIP_THRESHOLD = 0.85   # τ: skip non-discriminating problems
MIN_PROBLEMS   = 3      # minimum problems before any stopping decision

In [7]:
#Task-level sequential comparison
all_results = []
shared_tasks = sorted(set(samples_7b) & set(samples_11b))

for task in shared_tasks:
    oa = samples_11b[task]   # 11B outcomes
    ob = samples_7b[task]    # 7B outcomes
    post_a = BetaPosterior()
    post_b = BetaPosterior()
    stopped_at = len(oa)
    status = "exhausted"

    for i, (o11, o7) in enumerate(zip(oa, ob)):
        post_a.observe_one(o11)
        post_b.observe_one(o7)

        if i + 1 < MIN_PROBLEMS:
            continue

        if is_non_discriminating(post_a, post_b, SKIP_THRESHOLD):
            stopped_at = i + 1
            status = "skipped"
            break

        p = prob_a_beats_b(post_a, post_b)
        if p >= CONFIDENCE or p <= (1 - CONFIDENCE):
            stopped_at = i + 1
            status = "decided"
            break

    all_results.append({
        "task":       task,
        "post_a":     post_a,
        "post_b":     post_b,
        "stopped_at": stopped_at,
        "total_n":    len(oa),
        "p_a_beats_b": prob_a_beats_b(post_a, post_b),
        "status":     status,
    })

# summary
print(f"{'Task':<30} {'Stopped':>8} {'Total N':>8} {'Savings':>8} {'P(11B>7B)':>10} {'Status'}")
print("-" * 80)
total_stopped, total_n = 0, 0
for r in all_results:
    savings = (r["total_n"] - r["stopped_at"]) / r["total_n"] * 100
    total_stopped += r["stopped_at"]
    total_n       += r["total_n"]
    print(f"{r['task']:<30} {r['stopped_at']:>8,} {r['total_n']:>8,} "
          f"{savings:>7.1f}% {r['p_a_beats_b']:>10.3f}  {r['status']}")

total_savings = (total_n - total_stopped) / total_n * 100
print("-" * 80)
print(f"{'Total':<30} {total_stopped:>8,} {total_n:>8,} {total_savings:>7.1f}%")

Task                            Stopped  Total N  Savings  P(11B>7B) Status
--------------------------------------------------------------------------------
norbelebele                           4    4,500    99.9%      0.976  decided
norcommonsenseqa_nob                 24    4,990    99.5%      0.960  decided
noropenbookqa_nob                    65    1,870    96.5%      0.960  decided
nortruthfulqa_mc_nob                119    2,440    95.1%      0.954  decided
nrk_quiz_qa_nob                     198   18,000    98.9%      0.950  decided
--------------------------------------------------------------------------------
Total                               410   31,800    98.7%


In [8]:
# credibility intervals
from scipy import stats

print(f"{'Task':<30} {'Model':<6} {'Mean':>6}  {'95% Full-Corpus CrI'}")
print("-" * 68)

for task in sorted(set(samples_7b) & set(samples_11b)):
    for label, samples in [("11B", samples_11b[task]), ("7B", samples_7b[task])]:
        n = len(samples)
        s = sum(samples)
        # Jeffreys prior: Beta(0.5, 0.5)
        a = 0.5 + s
        b = 0.5 + n - s
        mean = a / (a + b)
        lo = stats.beta.ppf(0.025, a, b)
        hi = stats.beta.ppf(0.975, a, b)
        print(f"{task:<30} {label:<6} {mean:>6.3f}  [{lo:.3f}, {hi:.3f}]")
    print()

Task                           Model    Mean  95% Full-Corpus CrI
--------------------------------------------------------------------
norbelebele                    11B     0.486  [0.472, 0.501]
norbelebele                    7B      0.274  [0.261, 0.287]

norcommonsenseqa_nob           11B     0.662  [0.649, 0.676]
norcommonsenseqa_nob           7B      0.303  [0.291, 0.316]

noropenbookqa_nob              11B     0.706  [0.685, 0.727]
noropenbookqa_nob              7B      0.377  [0.355, 0.399]

nortruthfulqa_mc_nob           11B     0.554  [0.535, 0.574]
nortruthfulqa_mc_nob           7B      0.406  [0.386, 0.425]

nrk_quiz_qa_nob                11B     0.585  [0.578, 0.592]
nrk_quiz_qa_nob                7B      0.412  [0.404, 0.419]

